In [33]:
from thefuzz import process
import pycountry
import pandas as pd
import numpy as np

In [34]:
df =pd.read_excel(r'C:\Users\vaimi\Downloads\SUELDOS PROFESIONALES.xlsx')

In [35]:
df.head()

,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White
3,25-34,Nonprofits,Program Manager,NaN,62000,3000.0,USD,NaN,NaN,USA,Wisconsin,Milwaukee,8 - 10 years,5-7 years,College degree,Woman,White
4,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,60000,7000.0,USD,NaN,NaN,US,South Carolina,Greenville,8 - 10 years,5-7 years,College degree,Woman,White


In [36]:
#lista de paises validos
paises_validos = [country.name for country in pycountry.countries]

# Diccionario para siglas y casos especiales
siglas_map = {
    # ── United States ────────────────────────────────────────────────────────
    'Us': 'United States', 'Usa': 'United States',
    'U.S': 'United States', 'U.S.': 'United States',
    'U.S.A': 'United States', 'U.S.A.': 'United States',
    'U. S': 'United States', 'U. S.': 'United States',
    'Isa': 'United States', 'Is': 'United States',
    'Uss': 'United States', 'Uxz': 'United States',
    'Usaa': 'United States', 'Usab': 'United States',
    'Usat': 'United States', 'Usd': 'United States',
    'Us Of A': 'United States', 'The Us': 'United States',
    'America': 'United States', '🇺🇸': 'United States',
    'Virginia': 'United States', 'California': 'United States',
    'San Francisco': 'United States', 'Hartford': 'United States',
    'Puerto Rico': 'United States',
    'Us Govt Employee Overseas, Country Withheld': 'United States',
    'Usa Tomorrow': 'United States',
    'Usa-- Virgin Islands': 'United States',
    'Usa, But For Foreign Gov\'T': 'United States',
    'I Work For A Uae-Based Organization, Though I Am Personally In The Us.': 'United States',
    'Worldwide (Based In Us But Short Term Trips Aroudn The World)': 'United States',
    'United States (I Work From Home And My Clients Are All Over The Us/Canada/Pr': 'United States',

    # ── United Kingdom ───────────────────────────────────────────────────────
    'Uk': 'United Kingdom', 'U.K': 'United Kingdom',
    'U.K.': 'United Kingdom', 'Gb': 'United Kingdom',
    'England': 'United Kingdom', 'England, Gb': 'United Kingdom',
    'England, Uk': 'United Kingdom', 'England, Uk.': 'United Kingdom',
    'England, United Kingdom': 'United Kingdom',
    'England/Uk': 'United Kingdom', 'Englang': 'United Kingdom',
    'Britain': 'United Kingdom', 'Great Britain': 'United Kingdom',
    'Scotland': 'United Kingdom', 'Scotland, Uk': 'United Kingdom',
    'Wales': 'United Kingdom', 'Wales (Uk)': 'United Kingdom',
    'Wales (United Kingdom)': 'United Kingdom', 'Wales, Uk': 'United Kingdom',
    'Northern Ireland': 'United Kingdom',
    'Northern Ireland, United Kingdom': 'United Kingdom',
    'London': 'United Kingdom',
    'Uk (England)': 'United Kingdom', 'Uk (Northern Ireland)': 'United Kingdom',
    'U.K. (Northern England)': 'United Kingdom',
    'United Kingdom (England)': 'United Kingdom',
    'United Kingdom.': 'United Kingdom',
    'United Kindom': 'United Kingdom',
    'Unites Kingdom': 'United Kingdom',
    'Uk For U.S. Company': 'United Kingdom',
    'Uk, Remote': 'United Kingdom',
    'Uk, But For Globally Fully Remote Company': 'United Kingdom',
    'Isle Of Man': 'United Kingdom',
    'Jersey, Channel Islands': 'United Kingdom',
    'Loutreland': 'United Kingdom',

    # ── Canada ───────────────────────────────────────────────────────────────
    'Can': 'Canada', 'Canad': 'Canada', 'Canadá': 'Canada',
    'Canadw': 'Canada', 'Canda': 'Canada', 'Csnada': 'Canada',
    'Canada And Usa': 'Canada',
    'Canada, Ottawa, Ontario': 'Canada',
    'I Am Located In Canada But I Work For A Company In The Us': 'Canada',

    # ── Australia ────────────────────────────────────────────────────────────
    'Australi': 'Australia', 'Australian': 'Australia',

    # ── New Zealand ──────────────────────────────────────────────────────────
    'Nz': 'New Zealand',
    'Aotearoa New Zealand': 'New Zealand',
    'New Zealand Aotearoa': 'New Zealand',
    'From New Zealand But On Projects Across Apac': 'New Zealand',

    # ── Germany ──────────────────────────────────────────────────────────────
    'De': 'Germany',
    'Company In Germany. I Work From Pakistan.': 'Pakistan',

    # ── Argentina ────────────────────────────────────────────────────────────
    'I Work For An Us Based Company But I\'M From Argentina.': 'Argentina',
    'Argentina But My Org Is In Thailand': 'Thailand',

    # ── Netherlands ──────────────────────────────────────────────────────────
    'Nl': 'Netherlands', 'Nederland': 'Netherlands',
    'The Netherlands': 'Netherlands',

    # ── Czech Republic ───────────────────────────────────────────────────────
    'Czech Republic': 'Czech Republic',
    'Czechia': 'Czech Republic',
    'Česká Republika': 'Czech Republic',

    # ── Turkey ───────────────────────────────────────────────────────────────
    'Turkey': 'Turkey', 'Türkiye': 'Turkey', 'Turkiye': 'Turkey',

    # ── Otros nombres alternativos ───────────────────────────────────────────
    'Italia': 'Italy', 'Italy (South)': 'Italy',
    'Brasil': 'Brazil',
    'México': 'Mexico',
    'Panamá': 'Panama',
    'Danmark': 'Denmark',
    'Luxemburg': 'Luxembourg',
    'Burma': 'Myanmar', 'Mainland China': 'China',
    'Hong Kongkong': 'Hong Kong', 'Hong Konh': 'Hong Kong',
    'Catalonia': 'Spain',
    'Remote (Philippines)': 'Philippines',
    'From Romania, But For An Us Based Company': 'Romania',
    'Ibdia': 'India',
    'Ua': 'Ukraine',
    'Au': 'Australia',
    'Fr': 'France',
    'Uae': 'United Arab Emirates', 'U.A.E': 'United Arab Emirates',
    'United Arab Emirates': 'United Arab Emirates',

    # ── No utilizables → None ────────────────────────────────────────────────
    'Remote': None, 'Global': None, 'International': None,
    'Europe': None, 'Africa': None,
    'Na': None, 'N/A': None, 'N/A (Remote From Wherever I Want)': None,
    'Contracts': None, 'Policy': None,
    'Currently Finance': None, 'Dbfemf': None,
    'Ff': None, 'Ss': None, 'Uxz': None, 'Y': None,
    'Loutreland': None,
    '$2,175.84/Year Is Deducted For Benefits': None,
    'Bonus Based On Meeting Yearly Goals Set W/ My Supervisor': None,
    'We Don\'T Get Raises, We Get Quarterly Bonuses...': None,
    '(En Blanco)': None, '': None, '1': None,
}

# ── Normalizar claves a title case ───────────────────────────────────────────
siglas_map = {k.strip().title(): v for k, v in siglas_map.items()}


# Transformación
def limpiar_pais(valor, umbral=85):
    if pd.isna(valor) or str(valor).strip() == '':
        return np.nan
    
    valor_title = str(valor).strip().title()
    
    # Paso 1: buscar en diccionario con title() ← CORREGIDO
    if valor_title in siglas_map:
        resultado = siglas_map[valor_title]
        return resultado if resultado else np.nan
    
    # Paso 2: coincidencia exacta con lista de países
    if valor_title in paises_validos:
        return valor_title
    
    # Paso 3: fuzzy matching solo como último recurso
    resultado, similitud = process.extractOne(valor_title, paises_validos)
    
    if similitud >= umbral:
        return resultado
    else:
        return np.nan

In [37]:
df['country_clean'] = df['What country do you work in?'].apply(limpiar_pais)

In [38]:
# Valores originales vs valores limpios
comparacion = (
    df[['What country do you work in?', 'country_clean']]
    .drop_duplicates()
    .sort_values('country_clean')
    .reset_index(drop=True)
)

print(comparacion.to_string())

                                                                                                                                                                                          What country do you work in?                          country_clean
0                                                                                                                                                                                                          Afghanistan                            Afghanistan
1                                                                             I was brought in on this salary to help with the EHR and very quickly was promoted to current position but compensation was not altered.                    Antigua and Barbuda
2                                                                                                                                                               I work for an US based company but I'm from Argentina.                        

In [ ]:

df['currency_clean'] = df['Please indicate the currency'].str.strip().str.upper()

print(df['currency_clean'].value_counts())

df_clean = df[
    (df['currency_clean'] != 'OTHER') & 
    (df['country_clean'].notna())
].copy()


print(f"\nRegistros originales:    {len(df)}")
print(f"Registros eliminados:    {len(df) - len(df_clean)}")
print(f"Registros para análisis: {len(df_clean)}")

currency_clean
USD        23473
CAD         1676
GBP         1594
EUR          654
AUD/NZD      505
OTHER        171
CHF           37
SEK           37
JPY           23
ZAR           17
HKD            4
Name: count, dtype: int64

Registros originales:    28191
Registros eliminados:    191
Registros para análisis: 28000


In [ ]:
from forex_python.converter import CurrencyRates

c = CurrencyRates()

tasas_a_usd = {
    'USD':     1.0,
    'CAD':     c.get_rate('CAD', 'USD'),
    'GBP':     c.get_rate('GBP', 'USD'),
    'EUR':     c.get_rate('EUR', 'USD'),
    'AUD/NZD': c.get_rate('AUD', 'USD'),  
    'CHF':     c.get_rate('CHF', 'USD'),
    'SEK':     c.get_rate('SEK', 'USD'),
    'JPY':     c.get_rate('JPY', 'USD'),
    'ZAR':     c.get_rate('ZAR', 'USD'),
    'HKD':     c.get_rate('HKD', 'USD'),
}

print(tasas_a_usd)  

{'USD': 1.0, 'CAD': 0.732543, 'GBP': 1.333556, 'EUR': 1.1561, 'AUD/NZD': 0.700624, 'CHF': 1.278165, 'SEK': 0.108117, 'JPY': 0.006332, 'ZAR': 0.059816, 'HKD': 0.127887}


In [41]:
def convertir_a_usd(row, col_salario):
    moneda = row['currency_clean']
    salario = row[col_salario]

    if pd.isna(salario) or pd.isna(moneda):
        return np.nan
    if moneda in tasas_a_usd:
        return round(salario * tasas_a_usd[moneda], 2)
    return np.nan

In [42]:

col_salario = "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)"

col_extra = "How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits."

df_clean['salary_usd'] = df_clean.apply(
    lambda row: convertir_a_usd(row, col_salario), axis=1
)

df_clean['extra_usd'] = df_clean.apply(
    lambda row: convertir_a_usd(row, col_extra), axis=1
)

df_clean['total_compensation_usd'] = (
    df_clean['salary_usd'].fillna(0) + 
    df_clean['extra_usd'].fillna(0)
)

# ── Verificar ────────────────────────────────────────────────────────────────
print(df_clean[['currency_clean', 'salary_usd', 'extra_usd', 'total_compensation_usd']].head(10))

  currency_clean  salary_usd  extra_usd  total_compensation_usd
0            USD    55000.00       0.00                55000.00
1            GBP    72812.16    5334.22                78146.38
2            USD    34000.00        NaN                34000.00
3            USD    62000.00    3000.00                65000.00
4            USD    60000.00    7000.00                67000.00
5            USD    62000.00        NaN                62000.00
6            USD    33000.00    2000.00                35000.00
7            USD    50000.00        NaN                50000.00
8            USD   112000.00   10000.00               122000.00
9            USD    45000.00       0.00                45000.00


In [43]:
df_clean.columns

Index(['How old are you?', 'What industry do you work in?', 'Job title',
       'If your job title needs additional context, please clarify here:',
       'What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)',
       'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.',
       'Please indicate the currency',
       'If "Other," please indicate the currency here:',
       'If your income needs additional context, please provide it here:',
       'What country do you work in?',
       'If you're in the U.S., what state do you work in?',
       'What city do you work in?',
       'How many years of professional work experience do you have overall?',
       'How many years of pr

In [ ]:
df_clean = df_clean.rename(columns={

    'How old are you?':                                                                                                                                                                                                          'age',
    'What industry do you work in?':                                                                                                                                                                                             'industry',
    'Job title':                                                                                                                                                                                                                 'job_title',
    'If your job title needs additional context, please clarify here:':                                                                                                                                                          'job_title_context',
    'What is your annual salary? (You\'ll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)': 'annual_salary',
    'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.':                             'additional_compensation',
    'Please indicate the currency':                                                                                                                                                                                              'currency',
    'If "Other," please indicate the currency here:':                                                                                                                                                                            'currency_other',
    'If your income needs additional context, please provide it here:':                                                                                                                                                          'income_context',
    'What country do you work in?':                                                                                                                                                                                              'country',
    'If you\'re in the U.S., what state do you work in?':                                                                                                                                                                       'state',
    'What city do you work in?':                                                                                                                                                                                                 'city',
    'How many years of professional work experience do you have overall?':                                                                                                                                                       'experience_total',
    'How many years of professional work experience do you have in your field?':                                                                                                                                                 'experience_field',
    'What is your highest level of education completed?':                                                                                                                                                                        'education',
    'What is your gender?':                                                                                                                                                                                                      'gender',
    'What is your race? (Choose all that apply.)':                                                                                                                                                                               'race',


    'country_clean':            'country_clean',
    'currency_clean':           'currency_clean',
    'salary_usd':               'anual_salary_usd',
    'extra_usd':                'extra_usd',
    'total_compensation_usd':   'total_compensation_usd',
})


print(df_clean.columns.tolist())


['age', 'industry', 'job_title', 'job_title_context', 'annual_salary', 'additional_compensation', 'currency', 'currency_other', 'income_context', 'country', 'state', 'city', 'experience_total', 'experience_field', 'education', 'gender', 'race', 'country_clean', 'currency_clean', 'anual_salary_usd', 'extra_usd', 'total_compensation_usd']


In [45]:
print(df_clean['education'].value_counts(dropna=False))

education
College degree                        13468
Master's degree                        8848
Some college                           2070
PhD                                    1420
Professional degree (MD, JD, etc.)     1319
High School                             645
NaN                                     230
Name: count, dtype: int64


In [49]:
print(df_clean['job_title'].value_counts(dropna=False))

job_title
Software Engineer           303
Project Manager             250
Director                    219
Senior Software Engineer    205
Executive Assistant         183
                           ... 
Teaching assistant            1
Database Engineer             1
Analista de actuaria          1
actuarial analyst             1
Studio Director               1
Name: count, Length: 13400, dtype: int64


In [50]:
df_clean['job_title'] = df_clean['job_title'].str.title()

In [51]:
df_clean.head()

,age,industry,job_title,job_title_context,annual_salary,additional_compensation,currency,currency_other,income_context,country,...,experience_total,experience_field,education,gender,race,country_clean,currency_clean,anual_salary_usd,extra_usd,total_compensation_usd
0,25-34,Education (Higher Education),Research And Instruction Librarian,NaN,55000,0.0,USD,NaN,NaN,United States,...,5-7 years,5-7 years,Master's degree,Woman,White,United States,USD,55000.00,0.00,55000.00
1,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,54600,4000.0,GBP,NaN,NaN,United Kingdom,...,8 - 10 years,5-7 years,College degree,Non-binary,White,United Kingdom,GBP,72812.16,5334.22,78146.38
2,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,34000,NaN,USD,NaN,NaN,US,...,2 - 4 years,2 - 4 years,College degree,Woman,White,United States,USD,34000.00,NaN,34000.00
3,25-34,Nonprofits,Program Manager,NaN,62000,3000.0,USD,NaN,NaN,USA,...,8 - 10 years,5-7 years,College degree,Woman,White,United States,USD,62000.00,3000.00,65000.00
4,25-34,"Accounting, Banking & Finance",Accounting Manager,NaN,60000,7000.0,USD,NaN,NaN,US,...,8 - 10 years,5-7 years,College degree,Woman,White,United States,USD,60000.00,7000.00,67000.00


In [56]:
print(df_clean['race'].value_counts(dropna=False))

race
White                                                                                                                                                                   23186
Asian or Asian American                                                                                                                                                  1377
Black or African American                                                                                                                                                 691
Another option not listed here or prefer not to answer                                                                                                                    610
Hispanic, Latino, or Spanish origin                                                                                                                                       602
Hispanic, Latino, or Spanish origin, White                                                                                   

In [57]:

umbral = 60 


conteo = df_clean['race'].value_counts()


razas_poca_presencia = conteo[conteo < umbral].index.tolist()
print(f"Razas que se agruparán en 'Other': {razas_poca_presencia}")


df_clean['race_clean'] = df_clean['race'].apply(
    lambda x: 'Other' if x in razas_poca_presencia else x
)


print("\nDistribución final:")
print(df_clean['race_clean'].value_counts(dropna=False))

Razas que se agruparán en 'Other': ['Native American or Alaska Native', 'Black or African American, Hispanic, Latino, or Spanish origin', 'Asian or Asian American, Hispanic, Latino, or Spanish origin, White', 'Asian or Asian American, Hispanic, Latino, or Spanish origin', 'Black or African American, Hispanic, Latino, or Spanish origin, White', 'Hispanic, Latino, or Spanish origin, Native American or Alaska Native', 'Asian or Asian American, Another option not listed here or prefer not to answer', 'Hispanic, Latino, or Spanish origin, Native American or Alaska Native, White', 'Asian or Asian American, Black or African American', 'Asian or Asian American, Middle Eastern or Northern African', 'Black or African American, Native American or Alaska Native, White', 'Asian or Asian American, Black or African American, White', 'Asian or Asian American, White, Another option not listed here or prefer not to answer', 'Hispanic, Latino, or Spanish origin, Another option not listed here or prefer n

In [59]:
df_clean.to_excel('sueldos_limpio2.xlsx', index=False)
df_clean.to_csv('sueldos_limpio.csv', index=False, encoding='utf-8-sig')
print("✅ Archivos guardados")

✅ Archivos guardados
